# AI Code Auditor — Evaluation Notebook

Evaluates **baseline (zero-shot)** vs **fine-tuned (QLoRA)** CodeLlama-7B on the test set.

Produces:
- BLEU-4, ROUGE-L scores
- CWE classification accuracy
- Hallucination rate
- Qualitative failure case analysis
- Comparison table

### Before running:
1. GPU: T4 x1 is enough (inference only)
2. Upload `test.jsonl` as part of your dataset
3. Upload `lora_adapter_download.zip` as a separate Kaggle Dataset

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q \
    transformers==4.40.2 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    datasets==2.19.1 \
    sacrebleu \
    rouge-score \
    bitsandbytes==0.45.3
print('Done')

In [ ]:
# ── Cell 3: Paths ──────────────────────────────────────────────────────────
import os, json
from pathlib import Path

# Find test.jsonl
TEST_PATH = None
ADAPTER_PATH = None

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'test.jsonl':           TEST_PATH    = full
        if f == 'adapter_config.json':  ADAPTER_PATH = root

assert TEST_PATH,    'test.jsonl not found in /kaggle/input'
assert ADAPTER_PATH, 'adapter_config.json not found — upload lora_adapter as a dataset'

print(f'Test set   : {TEST_PATH}')
print(f'Adapter    : {ADAPTER_PATH}')

# Load test records
test_records = []
with open(TEST_PATH) as f:
    for line in f:
        test_records.append(json.loads(line.strip()))

# Use 100 samples for evaluation (full 731 would take too long)
EVAL_SAMPLES = 100
test_records = test_records[:EVAL_SAMPLES]
print(f'Evaluating : {len(test_records)} samples')
print(f'Sample CWE : {test_records[0]["cwe"]}')

In [ ]:
# ── Cell 4: Load base model (shared for both baseline + finetuned) ─────────
import os, torch
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = 'codellama/CodeLlama-7b-hf'

print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # better for generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
base_model.config.use_cache = True
print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

In [ ]:
# ── Cell 5: Inference helper ───────────────────────────────────────────────
import re

def build_prompt(vulnerable_code):
    return (
        '<s>[INST] <<SYS>>\n'
        'You are an expert security code auditor.\n'
        '<</SYS>>\n\n'
        f'Analyze the following C/C++ code for security vulnerabilities '
        f'and provide a secure rewrite:\n\n'
        f'```c\n{vulnerable_code}\n``` [/INST]'
    )

def run_inference(model, prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def extract_cwe(text):
    """Extract CWE ID from model output."""
    match = re.search(r'CWE-\d+', text)
    return match.group(0) if match else 'Unknown'

def extract_secure_code(text):
    """Extract code block from model output."""
    match = re.search(r'```(?:c|cpp)?\n(.*?)```', text, re.DOTALL)
    return match.group(1).strip() if match else text[:500]

print('Inference helpers ready')

In [ ]:
# ── Cell 6: Run BASELINE evaluation (zero-shot, no fine-tuning) ────────────
from tqdm import tqdm

print(f'Running baseline inference on {len(test_records)} samples...')
print('This will take ~20-30 minutes...')

baseline_results = []
for i, record in enumerate(tqdm(test_records)):
    prompt = build_prompt(record['vulnerable_code'])
    output = run_inference(base_model, prompt)

    baseline_results.append({
        'sample_id'            : i,
        'ground_truth_cwe'     : record['cwe'],
        'predicted_cwe'        : extract_cwe(output),
        'ground_truth_secure'  : record['secure_code'],
        'predicted_secure'     : extract_secure_code(output),
        'raw_output'           : output,
        'vulnerable_code'      : record['vulnerable_code'],
    })

# Save baseline results
with open('/kaggle/working/baseline_results.jsonl', 'w') as f:
    for r in baseline_results:
        f.write(json.dumps(r) + '\n')

print(f'Baseline done. Saved {len(baseline_results)} results.')

In [ ]:
# ── Cell 7: Load fine-tuned model ──────────────────────────────────────────
from peft import PeftModel

print(f'Loading LoRA adapter from {ADAPTER_PATH}...')
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
finetuned_model.eval()
print('Fine-tuned model ready.')

In [ ]:
# ── Cell 8: Run FINE-TUNED evaluation ─────────────────────────────────────
print(f'Running fine-tuned inference on {len(test_records)} samples...')
print('This will take ~20-30 minutes...')

finetuned_results = []
for i, record in enumerate(tqdm(test_records)):
    prompt = build_prompt(record['vulnerable_code'])
    output = run_inference(finetuned_model, prompt)

    finetuned_results.append({
        'sample_id'            : i,
        'ground_truth_cwe'     : record['cwe'],
        'predicted_cwe'        : extract_cwe(output),
        'ground_truth_secure'  : record['secure_code'],
        'predicted_secure'     : extract_secure_code(output),
        'raw_output'           : output,
        'vulnerable_code'      : record['vulnerable_code'],
    })

# Save finetuned results
with open('/kaggle/working/finetuned_results.jsonl', 'w') as f:
    for r in finetuned_results:
        f.write(json.dumps(r) + '\n')

print(f'Fine-tuned done. Saved {len(finetuned_results)} results.')

In [ ]:
# ── Cell 9: Compute metrics ────────────────────────────────────────────────
import sacrebleu
from rouge_score import rouge_scorer
import numpy as np, re
from collections import Counter

def compute_metrics(results, model_name):
    refs  = [r['ground_truth_secure'] for r in results]
    hyps  = [r['predicted_secure']    for r in results]
    gt_cwes   = [r['ground_truth_cwe'] for r in results]
    pred_cwes = [r['predicted_cwe']    for r in results]

    # BLEU-4
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    bleu4 = bleu.score

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rouge_scores = [scorer.score(r, h)['rougeL'].fmeasure for r, h in zip(refs, hyps)]
    rougeL = np.mean(rouge_scores)

    # CWE Top-1 accuracy
    cwe_acc = sum(1 for g, p in zip(gt_cwes, pred_cwes) if g == p) / len(gt_cwes)

    # Hallucination rate (secure rewrite contains dangerous patterns)
    danger_patterns = [r'\bstrcpy\s*\(', r'\bgets\s*\(', r'\bsprintf\s*\(']
    halluc = 0
    for r in results:
        orig_has = any(re.search(p, r['vulnerable_code']) for p in danger_patterns)
        pred_has = any(re.search(p, r['predicted_secure']) for p in danger_patterns)
        if not orig_has and pred_has:  # introduced new dangerous pattern
            halluc += 1
    halluc_rate = halluc / len(results)

    print(f'\n{"-"*45}')
    print(f'  {model_name}')
    print(f'{"-"*45}')
    print(f'  BLEU-4              : {bleu4:.2f}')
    print(f'  ROUGE-L             : {rougeL:.3f}')
    print(f'  CWE Top-1 Accuracy  : {cwe_acc:.3f} ({cwe_acc*100:.1f}%)')
    print(f'  Hallucination Rate  : {halluc_rate:.3f} ({halluc_rate*100:.1f}%)')
    print(f'  Samples evaluated   : {len(results)}')

    return {
        'model'         : model_name,
        'bleu4'         : round(bleu4, 2),
        'rougeL'        : round(rougeL, 3),
        'cwe_accuracy'  : round(cwe_acc, 3),
        'halluc_rate'   : round(halluc_rate, 3),
        'n_samples'     : len(results),
    }

baseline_metrics  = compute_metrics(baseline_results,  'Baseline (Zero-shot CodeLlama-7B)')
finetuned_metrics = compute_metrics(finetuned_results, 'Fine-tuned (QLoRA CodeLlama-7B)')

# Save metrics
with open('/kaggle/working/evaluation_metrics.json', 'w') as f:
    json.dump({'baseline': baseline_metrics, 'finetuned': finetuned_metrics}, f, indent=2)
print('\nMetrics saved to evaluation_metrics.json')

In [ ]:
# ── Cell 10: Comparison table + charts ────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

metrics_names  = ['BLEU-4', 'ROUGE-L', 'CWE Accuracy']
baseline_vals  = [baseline_metrics['bleu4'],  baseline_metrics['rougeL'],  baseline_metrics['cwe_accuracy']]
finetuned_vals = [finetuned_metrics['bleu4'], finetuned_metrics['rougeL'], finetuned_metrics['cwe_accuracy']]

x = np.arange(len(metrics_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AI Code Auditor — Baseline vs Fine-tuned Comparison', fontsize=13, fontweight='bold')

# Bar chart
ax = axes[0]
bars1 = ax.bar(x - width/2, baseline_vals,  width, label='Baseline (Zero-shot)', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, finetuned_vals, width, label='Fine-tuned (QLoRA)',   color='mediumseagreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylabel('Score')
ax.set_title('Performance Metrics')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', fontsize=9)

# Hallucination rate
ax2 = axes[1]
models = ['Baseline', 'Fine-tuned']
halluc = [baseline_metrics['halluc_rate'], finetuned_metrics['halluc_rate']]
colors = ['coral', 'mediumseagreen']
bars = ax2.bar(models, halluc, color=colors, alpha=0.85, width=0.4)
ax2.set_ylabel('Hallucination Rate')
ax2.set_title('Hallucination Rate (lower is better)')
ax2.grid(axis='y', alpha=0.3)
for bar in bars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{bar.get_height():.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.close()
print('Comparison chart saved.')

# Print improvement table
print('\n' + '='*55)
print(f'{"COMPARISON TABLE":^55}')
print('='*55)
print(f'{"Metric":<25} {"Baseline":>10} {"Fine-tuned":>12} {"Delta":>8}')
print('-'*55)
for m, b, f in zip(metrics_names, baseline_vals, finetuned_vals):
    delta = f - b
    arrow = '↑' if delta > 0 else '↓'
    print(f'{m:<25} {b:>10.3f} {f:>12.3f} {arrow}{abs(delta):>6.3f}')
print(f'{"Hallucination Rate":<25} {baseline_metrics["halluc_rate"]:>10.3f} {finetuned_metrics["halluc_rate"]:>12.3f}')
print('='*55)

In [ ]:
# ── Cell 11: Qualitative analysis — good & bad examples ───────────────────
print('='*60)
print('QUALITATIVE ANALYSIS')
print('='*60)

# Find cases where fine-tuned got CWE right but baseline didn't
improvements = [
    (b, f) for b, f in zip(baseline_results, finetuned_results)
    if b['ground_truth_cwe'] != b['predicted_cwe']
    and f['ground_truth_cwe'] == f['predicted_cwe']
]

# Find hallucination cases in fine-tuned
danger_patterns = [r'\bstrcpy\s*\(', r'\bgets\s*\(', r'\bsprintf\s*\(']
halluc_cases = [
    f for f in finetuned_results
    if not any(re.search(p, f['vulnerable_code']) for p in danger_patterns)
    and any(re.search(p, f['predicted_secure']) for p in danger_patterns)
]

print(f'\nCases where fine-tuned improved over baseline: {len(improvements)}')
print(f'Hallucination cases in fine-tuned            : {len(halluc_cases)}')

# Show top 3 improvement examples
print('\n--- TOP IMPROVEMENT EXAMPLES ---')
for i, (b, f) in enumerate(improvements[:3]):
    print(f'\nExample {i+1}:')
    print(f'  Ground truth CWE : {f["ground_truth_cwe"]}')
    print(f'  Baseline pred    : {b["predicted_cwe"]} ✗')
    print(f'  Fine-tuned pred  : {f["predicted_cwe"]} ✓')
    print(f'  Code snippet     : {f["vulnerable_code"][:100].strip()}...')

# Show top 2 failure/hallucination examples
if halluc_cases:
    print('\n--- HALLUCINATION EXAMPLES ---')
    for i, h in enumerate(halluc_cases[:2]):
        print(f'\nHallucination {i+1}:')
        print(f'  Ground truth CWE : {h["ground_truth_cwe"]}')
        print(f'  Predicted CWE    : {h["predicted_cwe"]}')
        print(f'  Issue: Secure rewrite still contains dangerous patterns')
        print(f'  Code snippet     : {h["vulnerable_code"][:100].strip()}...')

# Save qualitative report
qual_report = {
    'improvement_cases' : len(improvements),
    'hallucination_cases': len(halluc_cases),
    'top_improvements'  : [{
        'ground_truth_cwe': f['ground_truth_cwe'],
        'baseline_pred'   : b['predicted_cwe'],
        'finetuned_pred'  : f['predicted_cwe'],
        'code_snippet'    : f['vulnerable_code'][:200],
        'finetuned_output': f['raw_output'][:500],
    } for b, f in improvements[:5]],
    'hallucination_examples': [{
        'ground_truth_cwe': h['ground_truth_cwe'],
        'predicted_cwe'   : h['predicted_cwe'],
        'code_snippet'    : h['vulnerable_code'][:200],
        'bad_output'      : h['raw_output'][:500],
    } for h in halluc_cases[:3]],
}
with open('/kaggle/working/qualitative_report.json', 'w') as f:
    json.dump(qual_report, f, indent=2)
print('\nQualitative report saved.')

In [ ]:
# ── Cell 12: Save all outputs ──────────────────────────────────────────────
from pathlib import Path
import shutil

print('Files saved in /kaggle/working/:')
for p in sorted(Path('/kaggle/working').iterdir()):
    if p.is_file():
        print(f'  {p.name} ({p.stat().st_size/1e6:.1f} MB)')

print()
print('IMPORTANT: Click Save Version (top right) to persist!')
print()
print('Download these files:')
print('  evaluation_metrics.json  <- numbers for report')
print('  comparison_chart.png     <- chart for report')
print('  qualitative_report.json  <- examples for report')
print('  baseline_results.jsonl   <- raw baseline outputs')
print('  finetuned_results.jsonl  <- raw finetuned outputs')